In [7]:
import os
import re
from pathlib import Path

DATA_ROOT = Path("Data")

# === 关键词定义 ===
names = [
    "biodegradable", 
    "polylactic acid", "poly(lactic acid)", "poly(l-lactic acid)", "poly(lactide)",
    "poly(d-lactic acid)", "poly(d-lactide)", "poly(d,l-lactic acid)", 
    "poly(d,l-lactide)", "poly(lactide-co-glycolide)", 
    "polycaprolactone", "polybutylene succinate", "polyhydroxyalkanoates",
    "polyvinyl alcohol", "polycaprolactone-based", "polyhydroxyalkanoate-based",
    "chitosan", "cellulose", "alginate", "collagen", "gelatin", "silk fibroin", "hyaluronic acid", 
    "pectin", "starch", "chitin", "natural biodegradable", 
    "poly(butylene adipate)", "poly(butylene terephthalate)", "polycaprolactone-based polyester", 
    "poly(butylene succinate-co-adipate)", "poly(butylene succinate-co-terephthalate)",
    "poly(d,l-lactide-co-glycolide)", "polybutylene succinate-co-glutarate"
]

mechanical_keywords = [
    "young's modulus", "elastic modulus", "tensile modulus", 
    "tensile strength", "mechanical modulus", "elongation at break", "compressive strength", 
    "moduli", "stiffness", "flexural modulus", "bending modulus", "impact strength",
    "impact resistance", "stress", "strain", "mechanical properties", "mechanical performance", 
    "dynamic mechanical analysis", "fracture toughness", "toughness", "hardness", "flexibility",
    "rigidity", "compression strength", "creep resistance",
]

degradation_keywords = [
    "degradation rate", "biodegradation", "hydrolytic degradation", "enzymatic degradation", 
    "thermal degradation", "photo degradation", "degradability", "biodegradable behavior", 
    "mass loss", "weight loss", "molecular weight decrease", "degradation mechanism", 
    "degradation kinetics", "decomposition", "biodegradation performance", "environmental degradation", 
    "aging behavior",
]

# === 关键词匹配 ===
def find_matched_keywords(text, keyword_list):
    matched = []
    low = text.lower()
    compact = re.sub(r"[\s\-]+", "", low)
    for kw in keyword_list:
        pattern_str = re.escape(kw.lower()).replace(r"\ ", r"[\s\-]?")
        pattern = re.compile(r"\b" + pattern_str + r"\b", flags=re.IGNORECASE)
        if pattern.search(low) or kw.lower().replace(" ", "").replace("-", "") in compact:
            matched.append(kw)
    # 去重并保持顺序
    seen, dedup = set(), []
    for w in matched:
        if w not in seen:
            seen.add(w)
            dedup.append(w)
    return dedup

# === 结构解析 ===
def parse_txt_structure(text):
    blocks = {"Title": "", "Journal": "", "Date": "", "Abstract": "", "Paragraphs": []}
    title_match = re.search(r"Title:\s*(.*)", text)
    journal_match = re.search(r"Journal:\s*(.*)", text)
    date_match = re.search(r"Date:\s*(.*)", text)
    abstract_match = re.search(r"Abstract:\s*(.*?)\n(?:Paragraphs:|$)", text, flags=re.S)
    paragraphs_match = re.search(r"Paragraphs:\s*(.*)", text, flags=re.S)
    if title_match:
        blocks["Title"] = title_match.group(1).strip()
    if journal_match:
        blocks["Journal"] = journal_match.group(1).strip()
    if date_match:
        blocks["Date"] = date_match.group(1).strip()
    if abstract_match:
        blocks["Abstract"] = abstract_match.group(1).strip()
    if paragraphs_match:
        paras_text = paragraphs_match.group(1).strip()
        blocks["Paragraphs"] = [p.strip() for p in paras_text.split("\n\n") if p.strip()]
    return blocks

# === 仅按“命中窗口（前后各一段）且不重复”输出段落 + 摘要 ===
def process_txt_folder(input_folder, output_para_folder, output_abs_folder=None):
    os.makedirs(output_para_folder, exist_ok=True)
    if output_abs_folder is not None:
        os.makedirs(output_abs_folder, exist_ok=True)

    # === 新增：加载处理日志 ===
    # 将日志放在输出文件夹内，记录已经扫描过的原始文件名
    log_path = os.path.join(output_para_folder, ".processed_log.txt")
    processed_files = set()
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8") as log_f:
            processed_files = set(line.strip() for line in log_f if line.strip())

    txt_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".txt")]
    
    # 过滤掉已经处理过的文件
    files_to_process = [f for f in txt_files if f not in processed_files]
    
    print(f"🚀 总文件数: {len(txt_files)}，已跳过: {len(txt_files) - len(files_to_process)}，待处理: {len(files_to_process)}")

    def match_all(para):
        mn = find_matched_keywords(para, names)
        mm = find_matched_keywords(para, mechanical_keywords)
        md = find_matched_keywords(para, degradation_keywords)
        return mn, mm, md

    # 使用追加模式打开日志
    with open(log_path, "a", encoding="utf-8") as log_f:
        for fname in files_to_process:
            fpath = os.path.join(input_folder, fname)
            
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    text = f.read()

                blocks = parse_txt_structure(text)
                paragraphs = blocks["Paragraphs"]
                abstract = blocks["Abstract"]

                # === 先处理摘要命中 ===
                if output_abs_folder is not None and abstract:
                    mn_abs, mm_abs, md_abs = match_all(abstract)
                    has_abs_hit = bool(mn_abs) and (bool(mm_abs) or bool(md_abs))
                    if has_abs_hit:
                        header_abs = (
                            f"来源文件: {fname}\n"
                            f" Title: {blocks['Title']}\n"
                            f" Journal: {blocks['Journal']}\n"
                            f" Date: {blocks['Date']}\n"
                            f" 材料关键词: {', '.join(mn_abs) if mn_abs else '无'}\n"
                            f" 力学关键词: {', '.join(mm_abs) if mm_abs else '无'}\n"
                            f" 降解关键词: {', '.join(md_abs) if md_abs else '无'}\n\n"
                        )
                        body_abs = f"[Abstract]\n{abstract}\n"
                        abs_name = f"{os.path.splitext(fname)[0]}_Abstract.txt"
                        abs_path = os.path.join(output_abs_folder, abs_name)
                        with open(abs_path, "w", encoding="utf-8") as fa:
                            fa.write(header_abs + body_abs)

                # === 段落命中窗口 ===
                if not paragraphs:
                    print(f"⚠️ {fname} 未发现 Paragraphs 段落，跳过。")
                    # 即使没段落，也算处理过了
                    log_f.write(fname + "\n")
                    continue

                n = len(paragraphs)
                covered = set()
                hits_in_file = 0

                i = 0
                while i < n:
                    if i in covered:
                        i += 1
                        continue

                    mn, mm, md = match_all(paragraphs[i])
                    has_hit = bool(mn) and (bool(mm) or bool(md))
                    if not has_hit:
                        i += 1
                        continue

                    start = max(0, i - 1)
                    end = min(n - 1, i + 1)

                    agg_names, agg_mech, agg_degr = [], [], []
                    def extend_uniq(dst, src):
                        seen = set(dst)
                        for w in src:
                            if w not in seen:
                                dst.append(w)
                                seen.add(w)

                    for idx in range(start, end + 1):
                        mn_, mm_, md_ = match_all(paragraphs[idx])
                        extend_uniq(agg_names, mn_)
                        extend_uniq(agg_mech, mm_)
                        extend_uniq(agg_degr, md_)

                    header = (
                        f"来源文件: {fname}\n"
                        f"Title: {blocks['Title']}\nJournal: {blocks['Journal']}\nDate: {blocks['Date']}\n"
                        f"段编号范围: {start+1}-{end+1}\n"
                        f"材料关键词: {', '.join(agg_names) if agg_names else '无'}\n"
                        f"力学关键词: {', '.join(agg_mech) if agg_mech else '无'}\n"
                        f"降解关键词: {', '.join(agg_degr) if agg_degr else '无'}\n\n"
                    )

                    body = []
                    for j in range(start, end + 1):
                        tag = "[命中段落]" if j == i else "[上下文段落]"
                        body.append(f"{tag} 段{j+1}\n{paragraphs[j]}\n")
                    
                    combined = header + "\n".join(body)
                    out_name = f"{os.path.splitext(fname)[0]}_段{start+1}-{end+1}.txt"
                    out_path = os.path.join(output_para_folder, out_name)
                    with open(out_path, "w", encoding="utf-8") as fw:
                        fw.write(combined)

                    for idx in range(start, end + 1):
                        covered.add(idx)
                    hits_in_file += 1
                    i = end + 1

                # --- 写入处理日志 ---
                log_f.write(fname + "\n")
                log_f.flush() # 强制写入硬盘，防止崩溃

                if hits_in_file > 0:
                    print(f"✅ {fname} 命中 {hits_in_file} 个窗口")
                else:
                    print(f"⏭️ {fname} 已扫描，无命中")

            except Exception as e:
                print(f"❌ 处理出错 {fname}: {e}")

    print("\n🎯 全部文件筛选完成！")


if __name__ == "__main__":
    input_folder = DATA_ROOT / "RSC" / "txt"
    output_para_folder = DATA_ROOT / "RSC" / "hit_paragraphs"
    output_abs_folder = DATA_ROOT / "RSC" / "hit_abstracts"
    process_txt_folder(input_folder, output_para_folder, output_abs_folder)


🚀 总文件数: 372，已跳过: 372，待处理: 0

🎯 全部文件筛选完成！


In [3]:
# 只提取相关段落的--可能会缺失上下文信息

import os
import re

# === 关键词定义 ===
names = [
    "biodegradable", 
    "polylactic acid", "poly(lactic acid)", "poly(l-lactic acid)", "poly(lactide)",
    "poly(d-lactic acid)", "poly(d-lactide)", "poly(d,l-lactic acid)", 
    "poly(d,l-lactide)", "poly(lactide-co-glycolide)", 
    "polycaprolactone", "polybutylene succinate", "polyhydroxyalkanoates",
    "polyvinyl alcohol", "polycaprolactone-based", "polyhydroxyalkanoate-based",
    "chitosan", "cellulose", "alginate", "collagen", "gelatin", "silk fibroin", "hyaluronic acid", 
    "pectin", "starch", "chitin", "natural biodegradable", 
    "poly(butylene adipate)", "poly(butylene terephthalate)", "polycaprolactone-based polyester", 
    "poly(butylene succinate-co-adipate)", "poly(butylene succinate-co-terephthalate)",
    "poly(d,l-lactide-co-glycolide)", "polybutylene succinate-co-glutarate"
]

mechanical_keywords = [
    "young's modulus", "elastic modulus", "tensile modulus", 
    "tensile strength", "mechanical modulus", "elongation at break", "compressive strength", 
    "moduli", "stiffness", "flexural modulus", "bending modulus", "impact strength",
    "impact resistance", "stress", "strain", "mechanical properties", "mechanical performance", 
    "dynamic mechanical analysis", "fracture toughness", "toughness", "hardness", "flexibility",
    "rigidity", "compression strength", "creep resistance",
]

degradation_keywords = [
    "degradation rate", "biodegradation", "hydrolytic degradation", "enzymatic degradation", 
    "thermal degradation", "photo degradation", "degradability", "biodegradable behavior", 
    "mass loss", "weight loss", "molecular weight decrease", "degradation mechanism", 
    "degradation kinetics", "decomposition", "biodegradation performance", "environmental degradation", 
    "aging behavior",
]



# === 关键词匹配函数 ===
def find_matched_keywords(text, keyword_list):
    matched = []
    low = text.lower()
    compact = re.sub(r"[\s\-]+", "", low)
    for kw in keyword_list:
        pattern_str = re.escape(kw.lower()).replace(r"\ ", r"[\s\-]?")
        pattern = re.compile(r"\b" + pattern_str + r"\b", flags=re.IGNORECASE)
        if pattern.search(low) or kw.lower().replace(" ", "").replace("-", "") in compact:
            matched.append(kw)
    # 去重并保持原始顺序
    seen = set()
    deduped = []
    for w in matched:
        if w not in seen:
            seen.add(w)
            deduped.append(w)
    return deduped


# === 结构解析函数 ===
def parse_txt_structure(text):
    """识别并分割 Title / Journal / Date / Abstract / Paragraphs"""
    blocks = {"Title": "", "Journal": "", "Date": "", "Abstract": "", "Paragraphs": []}

    title_match = re.search(r"Title:\s*(.*)", text)
    journal_match = re.search(r"Journal:\s*(.*)", text)
    date_match = re.search(r"Date:\s*(.*)", text)
    abstract_match = re.search(r"Abstract:\s*(.*?)\n(?:Paragraphs:|$)", text, flags=re.S)
    paragraphs_match = re.search(r"Paragraphs:\s*(.*)", text, flags=re.S)

    if title_match:
        blocks["Title"] = title_match.group(1).strip()
    if journal_match:
        blocks["Journal"] = journal_match.group(1).strip()
    if date_match:
        blocks["Date"] = date_match.group(1).strip()
    if abstract_match:
        blocks["Abstract"] = abstract_match.group(1).strip()
    if paragraphs_match:
        paras_text = paragraphs_match.group(1).strip()
        blocks["Paragraphs"] = [p.strip() for p in paras_text.split("\n\n") if p.strip()]

    return blocks


# === 主流程：仅输出命中的“段落本身” ===
def process_txt_folder(input_folder, output_para_folder):
    os.makedirs(output_para_folder, exist_ok=True)
    txt_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".txt")]

    print(f"🚀 开始筛选（仅输出命中段），共 {len(txt_files)} 个文件。\n")

    for fname in txt_files:
        fpath = os.path.join(input_folder, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read()

        blocks = parse_txt_structure(text)
        if not blocks["Paragraphs"]:
            print(f"⚠️ {fname} 未发现 Paragraphs 段落，跳过。")
            continue

        paragraphs = blocks["Paragraphs"]
        hit_count = 0

        for i, para in enumerate(paragraphs):
            matched_names = find_matched_keywords(para, names)
            matched_mech = find_matched_keywords(para, mechanical_keywords)
            matched_degr = find_matched_keywords(para, degradation_keywords)

            # 命中条件：材料关键词 +（力学 或 降解）
            if matched_names and (matched_mech or matched_degr):
                hit_count += 1
                combined = (
                    f"来源文件: {fname}\n"
                    f"Title: {blocks['Title']}\nJournal: {blocks['Journal']}\nDate: {blocks['Date']}\n"
                    f"段编号: {i+1}\n"
                    f"材料关键词: {', '.join(matched_names)}\n"
                    f"力学关键词: {', '.join(matched_mech) if matched_mech else '无'}\n"
                    f"降解关键词: {', '.join(matched_degr) if matched_degr else '无'}\n\n"
                    f"[命中段落]\n{para}\n"
                )

                out_name = f"{os.path.splitext(fname)[0]}_段{i+1}.txt"
                out_path = os.path.join(output_para_folder, out_name)
                with open(out_path, "w", encoding="utf-8") as fw:
                    fw.write(combined)

        if hit_count > 0:
            print(f"✅ {fname} 命中 {hit_count} 段 → 已输出到 {output_para_folder}")
        else:
            print(f"⏭️ {fname} 无命中段，跳过输出。")

    print("\n🎯 全部文件筛选完成！")


if __name__ == "__main__":
    input_folder = DATA_ROOT / "Wiley" / "txt"           # 📂 输入TXT文件夹
    output_para_folder = DATA_ROOT / "Wiley" / "hit_paragraphs"  # 📂 命中段落输出（仅段本身）
    process_txt_folder(input_folder, output_para_folder)
    

🚀 开始筛选（仅输出命中段），共 297 个文件。

✅ 101002_macp200900441.txt 命中 5 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
⏭️ 101002_macp201000123.txt 无命中段，跳过输出。
⏭️ 101002_macp201000153.txt 无命中段，跳过输出。
✅ 101002_macp201000167.txt 命中 4 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
✅ 101002_macp201000491.txt 命中 6 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
✅ 101002_macp201000694.txt 命中 8 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
⏭️ 101002_macp201000720.txt 无命中段，跳过输出。
⏭️ 101002_macp201090036.txt 无命中段，跳过输出。
✅ 101002_macp201100042.txt 命中 2 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
⏭️ 101002_macp201100206.txt 无命中段，跳过输出。
⏭️ 101002_macp201100289.txt 无命中段，跳过输出。
✅ 101002_macp201100604.txt 命中 5 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
⏭️ 101002_macp201200073.txt 无命中段，跳过输出。
⏭️ 101002_macp201200576.txt 无命中段，跳过输出。
✅ 101002_macp201200612.txt 命中 4 段 → 已输出到 D:\FXR\1111-HTML\Wiley-Hit段落-single
⏭️ 101002_macp201300338.txt 无命中段，跳过输出。
⏭️ 101002_macp201300581.txt 无命中段，跳过输出。
⏭️ 101002_macp201300774.txt 无命中段，跳过输出。
✅ 101